# Coral Reef AI — Colab Resume After Local Training

This notebook **does not retrain** YOLO26m or EfficientNetV2-S.

It resumes from the two trained checkpoints and runs the remaining **video inference** stage:

```text
YOLO26m best checkpoint
        +
EfficientNetV2-S best checkpoint
        ↓
Detection → +10% Crop → Species
        ↓
Temporal Tracking + BBox EMA + Class EMA + Gap Hold
        ↓
Unknown / Uncertain rejection
        ↓
HQ H.264 Videos + CSV + Summary → Google Drive
```

## Put these files in Google Drive

```text
MyDrive/
└── Coral_Colab_Resume/
    ├── best_yolo26m.pt
    └── best_efficientnetv2s_macro_f1.pth
```

Video ZIP:

```text
MyDrive/Coral_video/coral_video.zip
```

The frozen detector confidence from the completed local validation is **0.375**.

The loop is resume-safe: a completed video is copied to Drive immediately and skipped on rerun.


In [ ]:
# CELL 1 — INSTALL + DRIVE + GPU

!pip -q install ultralytics==8.4.117 timm==1.0.22 scipy pandas

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc, csv, json, math, time, shutil, zipfile, subprocess

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import timm
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO
import ultralytics

if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime in Colab.')

DEVICE = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', torch.cuda.get_device_properties(0).total_memory / (1024**3))
print('Torch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
print('timm:', timm.__version__)
print('STATUS: ENVIRONMENT_READY')


In [ ]:
# CELL 2 — REQUIRED FILES ON DRIVE

MYDRIVE = Path('/content/drive/MyDrive')
RESUME_ROOT = MYDRIVE / 'Coral_Colab_Resume'
RESUME_ROOT.mkdir(parents=True, exist_ok=True)

YOLO_BEST = RESUME_ROOT / 'best_yolo26m.pt'
if not YOLO_BEST.exists() and (RESUME_ROOT / 'best.pt').exists():
    YOLO_BEST = RESUME_ROOT / 'best.pt'

BEST_CLS = RESUME_ROOT / 'best_efficientnetv2s_macro_f1.pth'
VIDEO_ZIP = MYDRIVE / 'Coral_video' / 'coral_video.zip'
DRIVE_VIDEO_OUT = RESUME_ROOT / 'Video_Results'
DRIVE_VIDEO_OUT.mkdir(parents=True, exist_ok=True)

for p in [YOLO_BEST, BEST_CLS, VIDEO_ZIP]:
    print(('OK   ' if p.exists() else 'MISS '), p)

missing = [p for p in [YOLO_BEST, BEST_CLS, VIDEO_ZIP] if not p.exists()]
if missing:
    raise FileNotFoundError('Missing:\n' + '\n'.join(str(p) for p in missing))

print('STATUS: FILES_READY')


In [ ]:
# CELL 3 — LOAD YOLO26m + EFFICIENTNETV2-S

YOLO_IMGSZ = 960
BEST_DET_CONF = 0.375
CONTEXT_PAD = 0.10

detector = YOLO(str(YOLO_BEST))

ckpt = torch.load(BEST_CLS, map_location=DEVICE, weights_only=False)
MODEL_NAME = ckpt.get('model_name', 'tf_efficientnetv2_s.in21k_ft_in1k')
class_to_idx = ckpt['class_to_idx']
idx_to_class = {int(v): str(k) for k, v in class_to_idx.items()}
NCLS = len(class_to_idx)

CLS_SIZE = int(ckpt.get('input_size', 300))
CONTEXT_PAD = float(ckpt.get('context_padding', CONTEXT_PAD))
MEAN = tuple(ckpt.get('mean', (0.5, 0.5, 0.5)))
STD = tuple(ckpt.get('std', (0.5, 0.5, 0.5)))

classifier = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=NCLS
).to(DEVICE)

classifier.load_state_dict(ckpt['model_state_dict'])
classifier.eval()

val_tf = transforms.Compose([
    transforms.Resize((CLS_SIZE, CLS_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print('YOLO:', YOLO_BEST)
print('YOLO imgsz/conf:', YOLO_IMGSZ, BEST_DET_CONF)
print('Classifier:', MODEL_NAME)
print('Classifier size:', CLS_SIZE)
print('Classes:', idx_to_class)
print('Checkpoint stage/epoch:', ckpt.get('stage'), ckpt.get('epoch'))
print('STATUS: MODELS_READY')


In [ ]:
# CELL 4 — TRACKER + CLASSIFIER HELPERS

TRACK_IOU = 0.20
MIN_HITS = 2
MAX_MISSED = 2
STATE_MAX_AGE = 20

BBOX_ALPHA = 0.45
VEL_ALPHA = 0.50
CLASS_ALPHA = 0.25

UNKNOWN_CONF = 0.55
UNKNOWN_MARGIN = 0.10

BOX_THICKNESS = 5
FONT_SCALE = 0.85
TEXT_THICKNESS = 2

VIDEO_CRF = 12
FFMPEG_PRESET = 'slow'
MAX_FRAMES_PER_VIDEO = None


def context_crop(im, box, pad=0.10):
    W, H = im.size
    x1, y1, x2, y2 = map(float, box)
    bw, bh = max(1., x2-x1), max(1., y2-y1)
    px, py = bw*pad, bh*pad
    return im.crop((
        max(0, int(math.floor(x1-px))),
        max(0, int(math.floor(y1-py))),
        min(W, int(math.ceil(x2+px))),
        min(H, int(math.ceil(y2+py))),
    ))


def cls_probs(crops):
    if not crops:
        return np.empty((0, NCLS), dtype=np.float32)
    batch = torch.stack([val_tf(c) for c in crops]).to(DEVICE, non_blocking=True)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        logits = classifier(batch)
        probs = torch.softmax(logits.float(), dim=1)
    out = probs.cpu().numpy().astype(np.float32)
    del batch, logits, probs
    return out


def iou_matrix(a, b):
    a = np.asarray(a, dtype=np.float32).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float32).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb-lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    aa = np.clip(a[:, 2]-a[:, 0], 0, None) * np.clip(a[:, 3]-a[:, 1], 0, None)
    bb = np.clip(b[:, 2]-b[:, 0], 0, None) * np.clip(b[:, 3]-b[:, 1], 0, None)
    return inter / np.clip(aa[:, None] + bb[None, :] - inter, 1e-9, None)


def clip_box(b, W, H):
    x1, y1, x2, y2 = map(float, b)
    return np.array([
        max(0, min(x1, W-1)),
        max(0, min(y1, H-1)),
        max(0, min(x2, W-1)),
        max(0, min(y2, H-1)),
    ], dtype=np.float32)


def new_track(tid, b, fi, dc):
    return {
        'id': tid,
        'box': np.asarray(b, dtype=np.float32).copy(),
        'vel': np.zeros(4, np.float32),
        'prob': None,
        'species': None,
        'cls_conf': 0.,
        'margin': 0.,
        'det_conf': float(dc),
        'last_seen': fi,
        'hits': 1,
        'missed': 0,
    }


def update_track(s, b, dc, fi):
    old = s['box'].copy()
    b = np.asarray(b, np.float32)
    new = BBOX_ALPHA*b + (1-BBOX_ALPHA)*old
    instant_vel = new-old
    s['vel'] = (VEL_ALPHA*instant_vel + (1-VEL_ALPHA)*s['vel']).astype(np.float32)
    s['box'] = new.astype(np.float32)
    s['det_conf'] = float(dc)
    s['last_seen'] = fi
    s['hits'] += 1
    s['missed'] = 0


def pred_box(s, ahead=1):
    return (s['box'] + s['vel']*float(ahead)).astype(np.float32)


def update_class(s, p):
    p = np.asarray(p, np.float32)
    sm = p.copy() if s['prob'] is None else CLASS_ALPHA*p + (1-CLASS_ALPHA)*s['prob']
    sm /= max(float(sm.sum()), 1e-9)
    s['prob'] = sm
    order = np.argsort(sm)[::-1]
    s['species'] = idx_to_class[int(order[0])]
    s['cls_conf'] = float(sm[order[0]])
    s['margin'] = float(sm[order[0]] - sm[order[1]])


def label_for(s):
    if s['cls_conf'] < UNKNOWN_CONF:
        return 'Unknown Coral'
    if s['margin'] < UNKNOWN_MARGIN:
        return 'Uncertain Coral'
    return s['species']


def associate(states, boxes):
    ids = list(states.keys())
    if not ids or len(boxes) == 0:
        return [], list(range(len(ids))), list(range(len(boxes))), ids

    predicted = np.stack([pred_box(states[tid]) for tid in ids])
    im = iou_matrix(predicted, boxes)
    rr, cc = linear_sum_assignment(1-im)

    matches, mt, md = [], set(), set()
    for r, c in zip(rr, cc):
        if im[r, c] < TRACK_IOU:
            continue
        matches.append((r, c, float(im[r, c])))
        mt.add(r)
        md.add(c)

    ut = [i for i in range(len(ids)) if i not in mt]
    ud = [i for i in range(len(boxes)) if i not in md]
    return matches, ut, ud, ids


def draw_track(frame, s, b, held=False):
    H, W = frame.shape[:2]
    b = clip_box(b, W, H)
    x1, y1, x2, y2 = [int(round(x)) for x in b]
    lab = label_for(s)

    if held:
        color = (0, 200, 255)
    elif lab in {'Unknown Coral', 'Uncertain Coral'}:
        color = (0, 165, 255)
    else:
        color = (0, 255, 0)

    cv2.rectangle(frame, (x1, y1), (x2, y2), color, BOX_THICKNESS)

    line1 = f"ID {s['id']} | {lab}"
    line2 = f"Det {s['det_conf']:.2f} | Cls {s['cls_conf']:.2f}"
    font = cv2.FONT_HERSHEY_SIMPLEX

    (w1, h1), _ = cv2.getTextSize(line1, font, FONT_SCALE, TEXT_THICKNESS)
    (w2, h2), _ = cv2.getTextSize(line2, font, FONT_SCALE, TEXT_THICKNESS)
    lw = max(w1, w2) + 18
    lh = h1 + h2 + 24
    yy = max(0, y1-lh)

    cv2.rectangle(frame, (x1, yy), (min(W-1, x1+lw), y1), color, -1)
    cv2.putText(frame, line1, (x1+8, yy+h1+6), font, FONT_SCALE, (0,0,0), TEXT_THICKNESS, cv2.LINE_AA)
    cv2.putText(frame, line2, (x1+8, yy+h1+h2+14), font, FONT_SCALE, (0,0,0), TEXT_THICKNESS, cv2.LINE_AA)


print('STATUS: HELPERS_READY')


In [ ]:
# CELL 5 — EXTRACT VIDEO ZIP

VWORK = Path('/content/YOLO26m_EfficientNetV2S_Resume')
VIN = VWORK / 'input'
VLOCAL = VWORK / 'output'

if VWORK.exists():
    shutil.rmtree(VWORK)

VIN.mkdir(parents=True, exist_ok=True)
VLOCAL.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(VIDEO_ZIP, 'r') as z:
    z.extractall(VIN)

exts = {'.mp4','.mov','.avi','.mkv','.m4v','.webm','.mpg','.mpeg'}
video_paths = sorted([
    p for p in VIN.rglob('*')
    if p.is_file() and p.suffix.lower() in exts
])

if not video_paths:
    raise RuntimeError('No videos found in coral_video.zip')

print('Videos:')
for i, p in enumerate(video_paths, 1):
    print(i, p.name)

print('STATUS: VIDEOS_READY')


In [ ]:
# CELL 6 — RUN / RESUME ALL VIDEOS

video_summaries = []

for vi, vp in enumerate(video_paths, 1):
    stem = vp.stem.replace(' ', '_')
    drive_dir = DRIVE_VIDEO_OUT / stem
    drive_summary = drive_dir / f'{stem}_summary.json'
    drive_video = drive_dir / f'{stem}_YOLO26m_EfficientNetV2S_HQ.mp4'

    if drive_summary.exists() and drive_video.exists():
        print(f'\nSKIP {vi}/{len(video_paths)}: {vp.name} — already complete in Drive')
        try:
            video_summaries.append(json.loads(drive_summary.read_text(encoding='utf-8')))
        except Exception:
            pass
        continue

    print('\n' + '='*110)
    print(f'VIDEO {vi}/{len(video_paths)}: {vp.name}')

    video_detector = YOLO(str(YOLO_BEST))
    cap = cv2.VideoCapture(str(vp))
    if not cap.isOpened():
        print('SKIP: OpenCV could not open video')
        continue

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    if not np.isfinite(fps) or fps <= 0:
        fps = 25.0

    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    local_dir = VLOCAL / stem
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)

    outv = local_dir / f'{stem}_YOLO26m_EfficientNetV2S_HQ.mp4'
    outcsv = local_dir / f'{stem}_tracks.csv'

    cmd = [
        'ffmpeg', '-loglevel', 'error', '-y',
        '-f', 'rawvideo', '-vcodec', 'rawvideo', '-pix_fmt', 'bgr24',
        '-s', f'{W}x{H}', '-r', f'{fps:.8f}', '-i', 'pipe:0',
        '-i', str(vp),
        '-map', '0:v:0', '-map', '1:a:0?',
        '-c:v', 'libx264', '-preset', FFMPEG_PRESET, '-crf', str(VIDEO_CRF),
        '-pix_fmt', 'yuv420p',
        '-c:a', 'aac', '-b:a', '192k',
        '-movflags', '+faststart', '-shortest',
        str(outv)
    ]

    ff = subprocess.Popen(
        cmd,
        stdin=subprocess.PIPE,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE
    )

    cf = open(outcsv, 'w', newline='', encoding='utf-8')
    cw = csv.DictWriter(cf, fieldnames=[
        'frame','time_sec','track_id','source',
        'x1','y1','x2','y2','det_conf',
        'species','cls_conf','class_margin'
    ])
    cw.writeheader()

    states = {}
    next_id = 1
    fi = -1
    unique = set()
    held_rows = 0
    det_ms, cls_ms, pipe_ms = [], [], []
    last_label = {}
    transitions = 0
    switches = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        fi += 1
        if MAX_FRAMES_PER_VIDEO is not None and fi >= MAX_FRAMES_PER_VIDEO:
            break

        ts = time.perf_counter()

        td = time.perf_counter()
        r = video_detector.predict(
            source=frame,
            imgsz=YOLO_IMGSZ,
            conf=BEST_DET_CONF,
            iou=0.70,
            max_det=500,
            device=0,
            verbose=False
        )[0]
        det_ms.append((time.perf_counter()-td)*1000)

        boxes = r.boxes.xyxy.detach().cpu().numpy().astype(np.float32).reshape(-1,4)
        conf = r.boxes.conf.detach().cpu().numpy().astype(np.float32).reshape(-1)

        matches, _, unmatched_dets, ids = associate(states, boxes)
        current = set()

        for ti, di, _ in matches:
            tid = ids[ti]
            update_track(states[tid], boxes[di], conf[di], fi)
            current.add(tid)

        for di in unmatched_dets:
            tid = next_id
            next_id += 1
            states[tid] = new_track(tid, boxes[di], fi, conf[di])
            current.add(tid)
            unique.add(tid)

        for tid in list(states):
            if tid in current:
                continue
            age = fi - states[tid]['last_seen']
            states[tid]['missed'] = age
            if age > STATE_MAX_AGE:
                del states[tid]

        tc = time.perf_counter()
        crop_ids, crops = [], []
        pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        for tid in current:
            b = clip_box(states[tid]['box'], W, H)
            crops.append(context_crop(pil, b, CONTEXT_PAD))
            crop_ids.append(tid)

        pp = cls_probs(crops)
        for tid, p in zip(crop_ids, pp):
            update_class(states[tid], p)

        cls_ms.append((time.perf_counter()-tc)*1000)

        for tid in sorted(current):
            if tid not in states or states[tid]['hits'] < MIN_HITS:
                continue

            s = states[tid]
            draw_track(frame, s, s['box'], held=False)
            b = s['box']
            lab = label_for(s)

            if lab not in {'Unknown Coral','Uncertain Coral'}:
                if tid in last_label:
                    transitions += 1
                    if lab != last_label[tid]:
                        switches += 1
                last_label[tid] = lab

            cw.writerow({
                'frame': fi,
                'time_sec': fi/fps,
                'track_id': tid,
                'source': 'DETECTED',
                'x1': float(b[0]), 'y1': float(b[1]),
                'x2': float(b[2]), 'y2': float(b[3]),
                'det_conf': s['det_conf'],
                'species': lab,
                'cls_conf': s['cls_conf'],
                'class_margin': s['margin']
            })

        for tid, s in list(states.items()):
            if tid in current or s['hits'] < MIN_HITS:
                continue
            if 1 <= s['missed'] <= MAX_MISSED:
                b = pred_box(s, s['missed'])
                draw_track(frame, s, b, held=True)
                held_rows += 1
                cw.writerow({
                    'frame': fi,
                    'time_sec': fi/fps,
                    'track_id': tid,
                    'source': 'HELD',
                    'x1': float(b[0]), 'y1': float(b[1]),
                    'x2': float(b[2]), 'y2': float(b[3]),
                    'det_conf': s['det_conf'],
                    'species': label_for(s),
                    'cls_conf': s['cls_conf'],
                    'class_margin': s['margin']
                })

        try:
            ff.stdin.write(frame.tobytes())
        except BrokenPipeError:
            ff.stdin.close()
            ff.wait()
            err = ff.stderr.read().decode('utf-8', errors='ignore')
            raise RuntimeError('FFmpeg failed:\n' + err[-5000:])

        pipe_ms.append((time.perf_counter()-ts)*1000)

        if (fi+1) % 100 == 0:
            recent = float(np.mean(pipe_ms[-100:]))
            print(
                f'Frames={fi+1} | IDs={len(unique)} | active={len(states)} '
                f'| held={held_rows} | {recent:.1f}ms ~{1000/recent:.2f} FPS'
            )

    cap.release()
    ff.stdin.close()
    ff.wait()
    ff_err = ff.stderr.read().decode('utf-8', errors='ignore')
    cf.close()

    if ff.returncode != 0:
        raise RuntimeError(ff_err[-5000:])

    summary = {
        'video': vp.name,
        'frames': fi+1,
        'unique_track_ids': len(unique),
        'held_rows': held_rows,
        'mean_detector_ms': float(np.mean(det_ms)) if det_ms else 0.,
        'mean_classifier_ms': float(np.mean(cls_ms)) if cls_ms else 0.,
        'mean_pipeline_ms': float(np.mean(pipe_ms)) if pipe_ms else 0.,
        'species_label_transitions': transitions,
        'species_label_switches': switches,
    }
    summary['pipeline_fps'] = (
        1000/summary['mean_pipeline_ms']
        if summary['mean_pipeline_ms'] else 0.
    )
    summary['species_flicker_rate'] = (
        switches/transitions if transitions else 0.
    )

    (local_dir / f'{stem}_summary.json').write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding='utf-8'
    )

    if drive_dir.exists():
        shutil.rmtree(drive_dir)
    shutil.copytree(local_dir, drive_dir)

    video_summaries.append(summary)
    print('DONE:', json.dumps(summary, indent=2))
    print('Saved:', drive_dir)

    del video_detector
    gc.collect()
    torch.cuda.empty_cache()

print('STATUS: VIDEO_TEST_COMPLETE')


In [ ]:
# CELL 7 — FINAL SUMMARY

all_summaries = []
for p in sorted(DRIVE_VIDEO_OUT.glob('*/*_summary.json')):
    try:
        all_summaries.append(json.loads(p.read_text(encoding='utf-8')))
    except Exception as e:
        print('Could not read', p, e)

video_df = pd.DataFrame(all_summaries)
summary_csv = DRIVE_VIDEO_OUT / 'all_video_summary.csv'
video_df.to_csv(summary_csv, index=False)

if len(video_df) and video_df['species_label_transitions'].sum() > 0:
    global_flicker = float(
        video_df['species_label_switches'].sum()
        / video_df['species_label_transitions'].sum()
    )
else:
    global_flicker = 0.0

mean_fps = float(video_df['pipeline_fps'].mean()) if len(video_df) else 0.0

LOCAL_VALIDATED_METRICS = {
    'detector': {
        'model': 'YOLO26m',
        'imgsz': 960,
        'mAP50_95': 0.5071313378906399,
        'mAP50': 0.8069212310784861,
        'mAP75': 0.5178718127673396,
        'precision': 0.8362681084143911,
        'recall': 0.7146892655367232,
    },
    'classifier': {
        'model': 'tf_efficientnetv2_s.in21k_ft_in1k',
        'input_size': 300,
        'context_padding': 0.10,
        'accuracy': 0.8898305084745762,
        'macro_precision': 0.881736740475285,
        'macro_recall': 0.8885518617463962,
        'macro_f1': 0.8838170966675897,
        'weighted_f1': 0.8903669559332067,
    },
    'end_to_end_validation': {
        'validation_images': 100,
        'validation_gt': 354,
        'predictions': 292,
        'matched_detections': 248,
        'correct_detection_and_species': 229,
        'detector_confidence': 0.375,
        'detector_precision': 0.8493150684931506,
        'detector_recall': 0.7005649717514124,
        'classifier_accuracy_on_matched': 0.9233870967741935,
        'end_to_end_species_precision': 0.7842465753424658,
        'end_to_end_species_recall': 0.6468926553672316,
        'end_to_end_species_f1': 0.7089783281733746,
        'classification_errors': 19,
        'localization_errors': 16,
        'detector_misses': 90,
    }
}

FINAL = {
    'pipeline': 'YOLO26m + 10% Context + EfficientNetV2-S',
    'detector_checkpoint': str(YOLO_BEST),
    'classifier_checkpoint': str(BEST_CLS),
    'validated_local_metrics': LOCAL_VALIDATED_METRICS,
    'videos_processed': int(len(video_df)),
    'mean_video_fps': mean_fps,
    'global_species_flicker_rate': global_flicker,
    'video_results_folder': str(DRIVE_VIDEO_OUT),
    'detector_held_out_test_used': False,
}

final_json = RESUME_ROOT / 'FINAL_PIPELINE_SUMMARY_COLAB.json'
final_json.write_text(
    json.dumps(FINAL, indent=2, ensure_ascii=False),
    encoding='utf-8'
)

print('='*110)
print('COLAB CONTINUATION COMPLETE')
print('Videos:', len(video_df))
print('Mean pipeline FPS:', mean_fps)
print('Global species flicker:', global_flicker)
print('Summary CSV:', summary_csv)
print('Final JSON:', final_json)
print('Results:', DRIVE_VIDEO_OUT)
print('Detector held-out TEST was NOT used.')
print('STATUS: COMPLETE')


## Resume after a Colab disconnect

Just rerun Cells **1 → 6**.

Cell 6 checks Google Drive before each video. If a video's `.mp4` and `_summary.json` already exist, it skips that video and continues with the next one.
